# Longest Winning Streaks in Tennis Matches

You are given a table of tennis players and their matches that they could either win (W) or lose (L). Find the longest streak of wins. A streak is a set of consecutive won matches of one player. The streak ends once a player loses their next match. Output the ID of the player or players and the length of the streak.

🌀 Trust me, this one will surely challenge you...! You'll learn Mutiple Ctes, Joins, Group by. Give it a try and share the output! 👇


In [0]:
%skip
%sql
CREATE TABLE ska_catalog2.bronze.players_results ( match_date DATE, match_result VARCHAR(1), player_id BIGINT);

INSERT INTO ska_catalog2.bronze.players_results (match_date, match_result, player_id) VALUES ('2023-01-01', 'W', 1), ('2023-01-02', 'W', 1), ('2023-01-03', 'L', 1), ('2023-01-04', 'W', 1), ('2023-01-01', 'L', 2), ('2023-01-02', 'W', 2), ('2023-01-03', 'W', 2), ('2023-01-04', 'W', 2), ('2023-01-05', 'L', 2), ('2023-01-01', 'W', 3), ('2023-01-02', 'W', 3), ('2023-01-03', 'W', 3), ('2023-01-04', 'W', 3), ('2023-01-05', 'L', 3);

In [0]:
%sql
SELECT * FROM ska_catalog2.bronze.players_results

In [0]:
%sql
WITH rankedMatches AS (
  SELECT 
    player_id,
    match_date,
    match_result,
    ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY match_date) - ROW_NUMBER() OVER (PARTITION BY player_id , match_result ORDER BY match_date) AS grp
  FROM ska_catalog2.bronze.players_results
  WHERE match_result = 'W'
),
winStreaks As (
  SELECT 
    player_id,
    count(*) AS streak_length
  FROM rankedMatches
  GROUP BY player_id , grp
),
longestStreaks AS (
  SELECT
    player_id,
    MAX(streak_length) AS max_streak
  FROM winStreaks
  GROUP BY player_id
),
longestOverallStreaks AS (
  SELECT MAX( max_streak) AS longest_streak FROM longestStreaks
)
SELECT 
  ls.player_id,
  ls.max_streak AS longest_streak
FROM longestStreaks ls
JOIN longestOverallStreaks lo
ON ls.max_streak = lo.longest_streak